In [0]:
# 1. Data Munging -
# 2. Programatically try to find couple of data patterns applying below EDA (File: logistics_source1)
# Apply inferSchema and toDF to create a DF and analyse the actual data.
# Analyse the schema, datatypes, columns etc.,
# Analyse the duplicate records count and summary of the dataframe.

logistics_df=spark.read.csv("/Volumes/usecasecatalog/usecaseschema/usecasevolume/logistics_source1.txt",inferSchema=True,header=True).toDF("shipment_id","first_name","last_name","age","role")

logistics_df.show()
logistics_df.printSchema() # column details as tree
print(logistics_df.dtypes) # column details as list
print(logistics_df.columns) # columns as list
logistics_df.describe().show() # basic summary - count/mean/stddiv/min/max
logistics_df.summary().show() # with percentile details
logistics_df.schema # struct type of column info

#dataframe
from pyspark.sql.window import Window
from pyspark.sql.functions import *

w_spec = Window.partitionBy("shipment_id").orderBy("shipment_id")
final_df = logistics_df.withColumn("row_number",row_number().over(w_spec))
final_df.filter("row_number=2").drop("row_number").show()

#sql
logistics_df.createOrReplaceTempView("logistics")
dup_qry='''select count(*),shipment_id from logistics 
group by shipment_id
having count(*) >1'''
spark.sql(dup_qry).show()

In [0]:
# a. Passive Data Munging - (File: logistics_source1 and logistics_source2)

# shipment_id is non-numericr
# age is not an integer

logistics_df1=spark.read.csv("/Volumes/usecasecatalog/usecaseschema/usecasevolume/logistics_source1.txt",inferSchema=True,header=True).toDF("shipment_id","first_name","last_name","age","role")

logistics_df2=spark.read.csv("/Volumes/usecasecatalog/usecaseschema/usecasevolume/logistics_source2.txt",inferSchema=True,header=True).toDF("shipment_id","first_name","last_name","age","role","hub_location","vehicle_type")

#logistics_df1.show(5)
#logistics_df2.show(5)

joined_df = logistics_df1.join(logistics_df2,logistics_df1.shipment_id==logistics_df2.shipment_id, how="inner").drop(logistics_df2.shipment_id,logistics_df2.first_name,logistics_df2.last_name,logistics_df2.age,logistics_df2.role)

joined_df.show(5)

from pyspark.sql.functions import col

filtered_df = joined_df.\
    filter(~col("shipment_id").cast("string").rlike("^[0-9]+$") \
    & (col("age")).cast("string").isNotNull())

filtered_df.show()



In [0]:
# b. Active Data Munging File: logistics_source1 and logistics_source2

# 1.Combining Data + Schema Merging (Structuring)

'''
both files without enforcing schema
Align them into a single canonical schema: shipment_id, first_name, last_name, age, role, hub_location, vehicle_type, data_source
Add data_source column with values as: system1, system2 in the respective dataframes
''' 

from pyspark.sql.functions import col,lit

logistics_df1=spark.read.csv("/Volumes/usecasecatalog/usecaseschema/usecasevolume/logistics_source1.txt",inferSchema="True",header=True)

logistics1= logistics_df1.withColumn("data_source",lit("source1"))


logistics_df2=spark.read.csv("/Volumes/usecasecatalog/usecaseschema/usecasevolume/logistics_source2.txt",inferSchema=True,header=True)

logistics2= logistics_df2.withColumn("data_source",lit("source2"))

combined_df = logistics1.unionByName(logistics2,allowMissingColumns=True)

# 2. Cleansing, Scrubbing:
'''
Cleansing (removal of unwanted datasets)

Mandatory Column Check - Drop any record where any of the following columns is NULL:shipment_id, role
Name Completeness Rule - Drop records where both of the following columns are NULL: first_name, last_name
Join Readiness Rule - Drop records where the join key is null: shipment_id
'''
#Scrubbing (convert raw to tidy)
'''
4. Age Defaulting Rule - Fill NULL values in the age column with: -1
5. Vehicle Type Default Rule - Fill NULL values in the vehicle_type column with: UNKNOWN
6. Invalid Age Replacement - Replace the following values in age: "ten" to -1 "" to -1
7. Vehicle Type Normalization - Replace inconsistent vehicle types: truck to LMV bike to TwoWheeler
'''

column_check_df = combined_df.na.drop(subset=["shipment_id","role"],how="any")

name_complete_df = column_check_df.na.drop(how="all",subset=["first_name","last_name"])

join_readiness_df = name_complete_df.na.drop(subset=["shipment_id"])


scrub1_df1 = join_readiness_df.na.fill(-1, subset=["age"]).\
                  na.fill("Unknown",subset=["vehicle_type"])

scrub1_df1.na.replace({"Truck":"LMV Vehicle","Bike":"Two wheeler"},subset=["vehicle_type"]).show(1000)

In [0]:
# 3. Standardization, De-Duplication and Replacement / Deletion of Data to make it in a usable format

#Creating shipments Details data Dataframe creation
#Create a DF by Reading Data from logistics_shipment_detail.json
# As this data is a clean json data, it doesn't require any cleansing or scrubbing.

from pyspark.sql.functions import lit,current_timestamp

shipment_df =spark.read.json("/Volumes/usecasecatalog/usecaseschema/usecasevolume/logistics_shipment_detail_3000.json",multiLine=True)

#Standardizations:
# 1.Add a column
#Source File: DF of logistics_shipment_detail_3000.json
#:domain as 'Logistics', current timestamp 'ingestion_timestamp' and 'False' as 'is_expedited'

shipment_add_column_df = shipment_df.\
                         withColumn("Domain",lit("Logistics")).\
                         withColumn("ingestion_timestamp",current_timestamp()).\
                         withColumn("is_expediated",lit("False"))  

display(shipment_add_column_df)  

# 2. Column Uniformity: role - Convert to lowercase
# Source File: DF of merged(logistics_source1 & logistics_source2)
# vehicle_type - Convert values to UPPERCASE
# Source Files: DF of logistics_shipment_detail_3000.json hub_location - Convert values to initcap case
# Source Files: DF of merged(logistics_source1 & logistics_source2)

from pyspark.sql.functions import col,lit,upper,initcap

logistics_df1=spark.read.csv("/Volumes/usecasecatalog/usecaseschema/usecasevolume/logistics_source1.txt",inferSchema="True",header=True)
logistics1= logistics_df1.withColumn("data_source",lit("source1"))


logistics_df2=spark.read.csv("/Volumes/usecasecatalog/usecaseschema/usecasevolume/logistics_source2.txt",inferSchema=True,header=True)

logistics2= logistics_df2.withColumn("data_source",lit("source2"))

combined_df=logistics1.unionByName(logistics2,allowMissingColumns=True)


upper_df=combined_df.\
    withColumn("Vehicle_type",upper("vehicle_type")).\
    withColumn("hub_location",initcap("hub_location"))


# 3. Format Standardization:
# Source Files: DF of logistics_shipment_detail_3000.json
# Convert shipment_date to yyyy-MM-dd
# Ensure shipment_cost has 2 decimal precision

from pyspark.sql.functions import to_date,round,col 
shipment_add_column_df1 = shipment_add_column_df.\
    withColumn("Shipment_date",to_date(col("Shipment_date"),"dd-MM-yy")).\
    withColumn("Shipment_cost",round(col("Shipment_cost"),2))    

# 4. Data Type Standardization
# Standardizing column data types to fix schema drift and enable mathematical operations.
# Source File: DF of merged(logistics_source1 & logistics_source2)
# age: Cast String to Integer
# Source File: DF of logistics_shipment_detail_3000.json
# shipment_weight_kg: Cast to Double
# Source File: DF of logistics_shipment_detail_3000.json
# is_expedited: Cast to Boolean

combined_df.printSchema() # already age is int
shipment_df.printSchema() # already shipment_weight_kg is double
shipment_add_column_df2= shipment_add_column_df1.withColumn("is_expediated",col("is_expediated").cast("boolean"))

# 5. Naming Standardization
# Source File: DF of merged(logistics_source1 & logistics_source2)
# Rename: first_name to staff_first_name
# Rename: last_name to staff_last_name
# Rename: hub_location to origin_hub_city

final_df = upper_df.withColumnRenamed("first_name", "staff_first_name").\
    withColumnRenamed("last_name", "staff_last_name").\
    withColumnRenamed("hub_location", "origin_hub_city") 

# 6. Reordering columns logically in a better standard format:
# Source File: DF of Data from all 3 files
# shipment_id (Identifier), staff_first_name (Dimension)staff_last_name (Dimension), role (Dimension), origin_hub_city (Location), shipment_cost (Metric), ingestion_timestamp (Audit)

final1_df = final_df.unionByName(shipment_add_column_df2,allowMissingColumns=True)

final2_df = final1_df.select("shipment_id","staff_first_name","staff_last_name","role","origin_hub_city","shipment_cost","ingestion_timestamp")

# Deduplication:
# Apply Record Level De-Duplication

final2_df.count() # 3151

final2_df.dropDuplicates().count() # 3148

# Apply Column Level De-Duplication (Primary Key Enforcement)

final2_df.dropDuplicates(["shipment_id"]).count() # 2719


In [0]:
# 2. Data Enrichment - Detailing of data

from pyspark.sql.functions import col,lit,upper,initcap,concat_ws

logistics_df1=spark.read.csv("/Volumes/usecasecatalog/usecaseschema/usecasevolume/logistics_source1.txt",inferSchema="True",header=True)
logistics1= logistics_df1.withColumn("data_source",lit("source1"))


logistics_df2=spark.read.csv("/Volumes/usecasecatalog/usecaseschema/usecasevolume/logistics_source2.txt",inferSchema=True,header=True)

logistics2= logistics_df2.withColumn("data_source",lit("source2"))

combined_df=logistics1.unionByName(logistics2,allowMissingColumns=True)

'''Adding of Columns (Data Enrichment)
Creating new derived attributes to enhance traceability and analytical capability.

1. Add Audit Timestamp (load_dt) Source File: DF of logistics_source1 and logistics_source2

Scenario: We need to track exactly when this record was ingested into our Data Lakehouse for auditing purposes.
Action: Add a column load_dt using the function current_timestamp().
2. Create Full Name (full_name) Source File: DF of logistics_source1 and logistics_source2

Scenario: The reporting dashboard requires a single field for the driver's name instead of separate columns.
Action: Create full_name by concatenating first_name and last_name with a space separator.
Result: "Rajesh" + " " + "Kumar" -> "Rajesh Kumar" '''


combined1_df=combined_df.withColumn("load_dt",current_timestamp()).\
            withColumn("full_name",concat_ws(" ","first_name","last_name")).drop("first_name","last_name")

combined1_df.select("shipment_id","full_name","role","age","hub_location","vehicle_type","load_dt","data_source").show()

'''
3. Define Route Segment (route_segment) Source File: DF of logistics_shipment_detail_3000.json

Scenario: The logistics team wants to analyze performance based on specific transport lanes (Source to Destination).
Action: Combine source_city and destination_city with a hyphen.
Result: "Chennai" + "-" + "Pune" -> "Chennai-Pune"
4. Generate Vehicle Identifier (vehicle_identifier) Source File: DF of logistics_shipment_detail_3000.json

Scenario: We need a unique tracking code that immediately tells us the vehicle type and the shipment ID.
Action: Combine vehicle_type and shipment_id to create a composite key.
Result: "Truck" + "_" + "500001" -> "Truck_500001"
'''

shipment_df =spark.read.json("/Volumes/usecasecatalog/usecaseschema/usecasevolume/logistics_shipment_detail_3000.json",multiLine=True)

display(shipment_df.withColumn("Source to destination",concat_ws("-",col("source_city"),col("destination_city"))).\
    withColumn("vehicle_identifier",concat_ws("_",col("vehicle_type"),col("shipment_id"))))

    


In [0]:
# Deriving of Columns (Time Intelligence)
# Extracting temporal features from dates to enable period-based analysis and reporting.
# Source File: logistics_shipment_detail_3000.json

"""
1. Derive Shipment Year (shipment_year)
Scenario: Management needs an annual performance report to compare growth year-over-year.
Action: Extract the year component from shipment_date.
Result: "2024-04-23" -> 2024
2. Derive Shipment Month (shipment_month)

Scenario: Analysts want to identify seasonal peaks (e.g., increased volume in December).
Action: Extract the month component from shipment_date.
Result: "2024-04-23" -> 4 (April)
3. Flag Weekend Operations (is_weekend)

Scenario: The Operations team needs to track shipments handled during weekends to calculate overtime pay or analyze non-business day capacity.
Action: Flag as 'True' if the shipment_date falls on a Saturday or Sunday.
4. Flag shipment status (is_expedited)

Scenario: The Operations team needs to track shipments is IN_TRANSIT or DELIVERED.
Action: Flag as 'True' if the shipment_status IN_TRANSIT or DELIVERED.
"""

shipment_df =spark.read.json("/Volumes/usecasecatalog/usecaseschema/usecasevolume/logistics_shipment_detail_3000.json",multiLine=True)

from pyspark.sql.functions import to_date,year,col,month,dayofweek,when


shipment_df1=shipment_df.withColumn("shipment_date",to_date(col("shipment_Date"),"yy-MM-dd")).\
              withColumn("shipment_year",year(col("shipment_date"))).\
              withColumn("shipment_month",month(col("shipment_date"))).\
              withColumn("Weekday",dayofweek(col("shipment_date"))).\
              withColumn("is_weekend",when(col("Weekday").isin(1,7),"True").otherwise("False")).\
              withColumn("is_expediated",when(col("shipment_status").isin("IN_TRANSIT","DELIVERED"),"True").otherwise("False"))           

shipment_df1.show()

Enrichment/Business Logics (Calculated Fields)
Deriving new metrics and financial indicators using mathematical and date-based operations.
Source File: logistics_shipment_detail_3000.json